In [2]:
import pandas as pd
import numpy as np

In [3]:
bookings_data = pd.read_csv("/Users/john/Desktop/Group AM ML Project/hotel_bookings.csv")
bookings_data

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,City Hotel,0,23,2017,August,35,30,2,5,2,...,No Deposit,394.0,NaN,0,Transient,96.14,0,0,Check-Out,2017-09-06
119386,City Hotel,0,102,2017,August,35,31,2,5,3,...,No Deposit,9.0,NaN,0,Transient,225.43,0,2,Check-Out,2017-09-07
119387,City Hotel,0,34,2017,August,35,31,2,5,2,...,No Deposit,9.0,NaN,0,Transient,157.71,0,4,Check-Out,2017-09-07
119388,City Hotel,0,109,2017,August,35,31,2,5,2,...,No Deposit,89.0,NaN,0,Transient,104.40,0,0,Check-Out,2017-09-07


### Developing a Logistic Regression Model

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

from feature_eng import feature_engineering

In [5]:
# Split X/y (target NOT in feature_engineering)
y = bookings_data["is_canceled"]

X = bookings_data.drop(columns=["is_canceled"])

feat = FunctionTransformer(feature_engineering, validate=False)

numeric_cols = [
    "lead_time",
    "arrival_date_year",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "booking_changes",
    "total_of_special_requests",
    "adr",
    "required_car_parking_spaces",
    "days_in_waiting_list",
    "is_repeated_guest",
    "total_nights",
    "total_guests",
    "arrival_month"
]

categorical_cols = [
    "meal",
    "reserved_room_type",
    "deposit_type",
    "market_segment",
    "distribution_channel",
    "customer_type",
    "hotel"
]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


cat_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])


preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, numeric_cols),
        ("cat", cat_pipeline, categorical_cols)
    ],
    remainder="drop"
)


# Create the full pipeline with preprocessing + feature engineering + model
logreg_pipeline = Pipeline([
    ("feature_engineering", feat),
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])


# Train-test split
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.33, random_state=7)


# Fit the pipeline
logreg_pipeline.fit(X_train, Y_train)


Pipeline(steps=[('feature_engineering',
                 FunctionTransformer(func=<function feature_engineering at 0x17892e020>)),
                ('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['lead_time',
                                                   'arrival_date_year',
                                                   'stays_in_weekend_nights',
                                                   'stays_in_week_nights',
                                                   'previous_cance...
                                                   'required_car_parking_spaces',
                                                   'days_in_waiting_list',
                                                   'is_repeated_guest',
                                                   'total_nights',
                                                   'total_guests',
                                                   'arrival_month']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['meal', 'reserved_room_type',
                                                   'deposit_type',
                                                   'market_segment',
                                                   'distribution_channel',
                                                   'customer_type',
                                                   'hotel'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [8]:
## Testing
from sklearn.metrics import confusion_matrix, f1_score

y_pred = logreg_pipeline.predict(X_test)

y_prob = logreg_pipeline.predict_proba(X_test)[:,1]

print(classification_report(Y_test, y_pred))

print("ROC AUC:", roc_auc_score(Y_test, y_prob))

print("Confusion Matrix:\n", confusion_matrix(Y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.82      0.83     24746
           1       0.71      0.72      0.72     14653

    accuracy                           0.79     39399
   macro avg       0.77      0.77      0.77     39399
weighted avg       0.79      0.79      0.79     39399

ROC AUC: 0.8537991403152169
Confusion Matrix:
 [[20386  4360]
 [ 4066 10587]]


### Logistic Regression Hyperparameter Tuning

In [9]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

## Defining the Hyperparameters to tune

param_grid = [
    {
        "model__penalty": ["l2"],
        "model__solver": ["lbfgs", "liblinear"],
        "model__C": [0.01, 0.1, 1, 10],
        "model__class_weight": [None, "balanced"]
    },
    {
        "model__penalty": ["l1"],
        "model__solver": ["liblinear"],
        "model__C": [0.01, 0.1, 1, 10],
        "model__class_weight": [None, "balanced"]
    }
]


logreg_pipeline = Pipeline([
    ("feature_engineering", feat),
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000))
])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

grid_search = GridSearchCV(
    estimator=logreg_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, Y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=7, shuffle=True),
             estimator=Pipeline(steps=[('feature_engineering',
                                        FunctionTransformer(func=<function feature_engineering at 0x17892e020>)),
                                       ('preprocess',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['lead_time',
                                                                          'a...
                                                                          'hotel'])])),
                                       ('model',
                                        LogisticRegression(max_iter=2000))]),
             n_jobs=-1,
             param_grid=[{'model__C': [0.01, 0.1, 1, 10],
                          'model__class_weight': [None, 'balanced'],
                          'model__penalty': ['l2'],
                          'model__solver': ['lbfgs', 'liblinear']},
                         {'model__C': [0.01, 0.1, 1, 10],
                          'model__class_weight': [None, 'balanced'],
                          'model__penalty': ['l1'],
                          'model__solver': ['liblinear']}],
             scoring='f1', verbose=1)

In [10]:
## Inspecting the best model

print("Best params:", grid_search.best_params_)
print("Best CV F1_Score:", grid_search.best_score_)

Best params: {'model__C': 1, 'model__class_weight': 'balanced', 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Best CV F1_Score: 0.7123961175373443


In [12]:
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Test ROC_AUC_SCORE:", roc_auc_score(Y_test, y_prob))
print("\nClassification Report:\n")
print(classification_report(Y_test, y_pred))

print("Confusion Matrix:\n", confusion_matrix(Y_test, y_pred))

Test ROC_AUC_SCORE: 0.8537991403152169

Classification Report:

              precision    recall  f1-score   support

           0       0.83      0.82      0.83     24746
           1       0.71      0.72      0.72     14653

    accuracy                           0.79     39399
   macro avg       0.77      0.77      0.77     39399
weighted avg       0.79      0.79      0.79     39399

Confusion Matrix:
 [[20386  4360]
 [ 4066 10587]]


### Developing a Random Forest Model

In [13]:
from sklearn.ensemble import RandomForestClassifier

rf_num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


rf_cat_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

rf_preprocess = ColumnTransformer(
    transformers=[
        ("num", rf_num_pipeline, numeric_cols),
        ("cat", rf_cat_pipeline, categorical_cols),
    ],
    remainder="drop"
)

rf_pipeline = Pipeline([
    ("feature_engineering", feat),
    ("preprocess", rf_preprocess),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=7,
        n_jobs=-1
    ))
])

# Fit the RandomForest model
rf_pipeline.fit(X_train, Y_train)

Pipeline(steps=[('feature_engineering',
                 FunctionTransformer(func=<function feature_engineering at 0x17892e020>)),
                ('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['lead_time',
                                                   'arrival_date_year',
                                                   'stays_in_weekend_nights',
                                                   'stays_in_week_nights',
                                                   'previous_cance...
                                                   'arrival_month']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['meal', 'reserved_room_type',
                                                   'deposit_type',
                                                   'market_segment',
                                                   'distribution_channel',
                                                   'customer_type',
                                                   'hotel'])])),
                ('model',
                 RandomForestClassifier(class_weight='balanced_subsample',
                                        min_samples_leaf=5,
                                        min_samples_split=10, n_estimators=200,
                                        n_jobs=-1, random_state=7))])

In [14]:
# Evaluate RandomForest on the test set
rf_pred = rf_pipeline.predict(X_test)
rf_prob = rf_pipeline.predict_proba(X_test)[:, 1]

print("RandomForest ROC-AUC:", roc_auc_score(Y_test, rf_prob))
print("\nClassification Report:\n")
print(classification_report(Y_test, rf_pred))

print("Confusion Matrix:\n", confusion_matrix(Y_test, rf_pred))

RandomForest ROC-AUC: 0.9183952718577961

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.89      0.88     24746
           1       0.81      0.77      0.79     14653

    accuracy                           0.85     39399
   macro avg       0.84      0.83      0.83     39399
weighted avg       0.85      0.85      0.85     39399

Confusion Matrix:
 [[22095  2651]
 [ 3401 11252]]


### Random Forest Hyperparameter Tuning

In [15]:
from sklearn.model_selection import RandomizedSearchCV

## Define the search space

param_dist = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 5],
    "model__max_features": ["sqrt", "log2"],
    "model__class_weight": [None, "balanced", "balanced_subsample"]
}

## Randomized search

rf_pipeline = Pipeline([
    ("feature_engineering", feat),
    ("preprocess", rf_preprocess),
    ("model", RandomForestClassifier(random_state=7, n_jobs=-1))
])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    scoring="f1",
    cv=cv,
    random_state=7,
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, Y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=7, shuffle=True),
                   estimator=Pipeline(steps=[('feature_engineering',
                                              FunctionTransformer(func=<function feature_engineering at 0x17892e020>)),
                                             ('preprocess',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               ['lead_ti...
                                              RandomForestClassifier(n_jobs=-1,
                                                                     random_state=7))]),
                   n_jobs=-1,
                   param_distributions={'model__class_weight': [None,
                                                                'balanced',
                                                                'balanced_subsample'],
                                        'model__max_depth': [None, 10, 20, 30],
                                        'model__max_features': ['sqrt', 'log2'],
                                        'model__min_samples_leaf': [1, 2, 5],
                                        'model__min_samples_split': [2, 5, 10],
                                        'model__n_estimators': [100, 200, 300]},
                   random_state=7, scoring='f1', verbose=1)

In [16]:
## Inspecting the best RandomForest

print("Best RF params:", rf_search.best_params_)
print("Best RF CV F1_SCORE:", rf_search.best_score_)

Best RF params: {'model__n_estimators': 100, 'model__min_samples_split': 2, 'model__min_samples_leaf': 2, 'model__max_features': 'log2', 'model__max_depth': 30, 'model__class_weight': 'balanced_subsample'}
Best RF CV F1_SCORE: 0.7918786889238366


In [18]:
## Evaluate on the test set

best_rf = rf_search.best_estimator_

rf_pred = best_rf.predict(X_test)
rf_prob = best_rf.predict_proba(X_test)[:, 1]

print("RandomForest Test ROC-AUC:", roc_auc_score(Y_test, rf_prob))
print("\nClassification Report:\n")
print(classification_report(Y_test, rf_pred))
print("Confusion Matrix:\n", confusion_matrix(Y_test, rf_pred))

RandomForest Test ROC-AUC: 0.9228445976658922

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.91      0.89     24746
           1       0.83      0.77      0.80     14653

    accuracy                           0.85     39399
   macro avg       0.85      0.84      0.84     39399
weighted avg       0.85      0.85      0.85     39399

Confusion Matrix:
 [[22419  2327]
 [ 3408 11245]]


### Developing a Gradient Boost Model

In [19]:
from sklearn.ensemble import GradientBoostingClassifier

gb_num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
] )

gb_cat_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

gb_preprocess = ColumnTransformer(
    transformers=[
        ("num", gb_num_pipeline, numeric_cols),
        ("cat", gb_cat_pipeline, categorical_cols),
    ],
    remainder="drop"
)

gb_pipeline = Pipeline([
    ("feature_engineering", feat),
    ("preprocess", gb_preprocess),
    ("model", GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=7
    ))
])

# Fit and evaluate Gradient Boosting
gb_pipeline.fit(X_train, Y_train)

Pipeline(steps=[('feature_engineering',
                 FunctionTransformer(func=<function feature_engineering at 0x17892e020>)),
                ('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['lead_time',
                                                   'arrival_date_year',
                                                   'stays_in_weekend_nights',
                                                   'stays_in_week_nights',
                                                   'previous_cance...
                                                   'is_repeated_guest',
                                                   'total_nights',
                                                   'total_guests',
                                                   'arrival_month']),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['meal', 'reserved_room_type',
                                                   'deposit_type',
                                                   'market_segment',
                                                   'distribution_channel',
                                                   'customer_type',
                                                   'hotel'])])),
                ('model',
                 GradientBoostingClassifier(learning_rate=0.05,
                                            n_estimators=200,
                                            random_state=7))])

In [20]:
## Evaluate on the test set
gb_pred = gb_pipeline.predict(X_test)
gb_prob = gb_pipeline.predict_proba(X_test)[:, 1]

print("Gradient Boosting ROC-AUC:", roc_auc_score(Y_test, gb_prob))
print("\nClassification Report:\n")
print(classification_report(Y_test, gb_pred))
print("Confusion Matrix:\n", confusion_matrix(Y_test, gb_pred))

Gradient Boosting ROC-AUC: 0.8814560548563151

Classification Report:

              precision    recall  f1-score   support

           0       0.80      0.94      0.87     24746
           1       0.86      0.61      0.71     14653

    accuracy                           0.82     39399
   macro avg       0.83      0.77      0.79     39399
weighted avg       0.82      0.82      0.81     39399

Confusion Matrix:
 [[23252  1494]
 [ 5712  8941]]


### Gradient Boost Hyperparameter Tuning

In [21]:
# Hyperparameter search space
gb_param_dist = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__learning_rate": [0.01, 0.03, 0.05, 0.1],
    "model__max_depth": [2, 3, 4, 5],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__subsample": [0.6, 0.8, 1.0],
    "model__max_features": [None, "sqrt", "log2"]
}

gb_tuning_pipeline = Pipeline([
    ("feature_engineering", feat),
    ("preprocess", gb_preprocess),
    ("model", GradientBoostingClassifier(random_state=7))
])

gb_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

gb_search = RandomizedSearchCV(
    estimator=gb_tuning_pipeline,
    param_distributions=gb_param_dist,
    n_iter=20,
    scoring="f1",
    cv=gb_cv,
    random_state=7,
    n_jobs=-1,
    verbose=1
)

# Fit search
gb_search.fit(X_train, Y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


RandomizedSearchCV(cv=StratifiedKFold(n_splits=5, random_state=7, shuffle=True),
                   estimator=Pipeline(steps=[('feature_engineering',
                                              FunctionTransformer(func=<function feature_engineering at 0x17892e020>)),
                                             ('preprocess',
                                              ColumnTransformer(transformers=[('num',
                                                                               Pipeline(steps=[('imputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('scaler',
                                                                                                StandardScaler())]),
                                                                               ['lead_ti...
                                              GradientBoostingClassifier(random_state=7))]),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'model__learning_rate': [0.01, 0.03,
                                                                 0.05, 0.1],
                                        'model__max_depth': [2, 3, 4, 5],
                                        'model__max_features': [None, 'sqrt',
                                                                'log2'],
                                        'model__min_samples_leaf': [1, 2, 4, 8],
                                        'model__min_samples_split': [2, 5, 10,
                                                                     20],
                                        'model__n_estimators': [100, 200, 300,
                                                                500],
                                        'model__subsample': [0.6, 0.8, 1.0]},
                   random_state=7, scoring='f1', verbose=1)

In [22]:
### Investigating the best model
print("Best GradientBoost params:", gb_search.best_params_)
print("Best GradientBoost CV F1:", gb_search.best_score_)

Best GradientBoost params: {'model__subsample': 0.8, 'model__n_estimators': 500, 'model__min_samples_split': 5, 'model__min_samples_leaf': 8, 'model__max_features': 'sqrt', 'model__max_depth': 4, 'model__learning_rate': 0.05}
Best GradientBoost CV F1: 0.7345146919706783


In [24]:
### Evaluating on a test set

best_gb = gb_search.best_estimator_
gb_pred = best_gb.predict(X_test)
gb_prob = best_gb.predict_proba(X_test)[:, 1]

print("Tuned GradientBoost Test ROC-AUC:", roc_auc_score(Y_test, gb_prob))
print("\nClassification Report:\n")
print(classification_report(Y_test, gb_pred))
print("Confusion Matrix:\n", confusion_matrix(Y_test, gb_pred))

Tuned GradientBoost Test ROC-AUC: 0.8955737319074166

Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.94      0.87     24746
           1       0.86      0.65      0.74     14653

    accuracy                           0.83     39399
   macro avg       0.84      0.79      0.80     39399
weighted avg       0.83      0.83      0.82     39399

Confusion Matrix:
 [[23158  1588]
 [ 5183  9470]]


### Investigating the most predictive features of the best model: Tuned Random Forest

In [25]:
best_rf = rf_search.best_estimator_

rf_model = best_rf.named_steps["model"]
preprocessor = best_rf.named_steps["preprocess"]

feature_names = preprocessor.get_feature_names_out()
importances = rf_model.feature_importances_

rf_imp_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

rf_imp_df.head(20)

,Feature,Importance
0,num__lead_time,0.141664
31,cat__deposit_type_Non Refund,0.098930
30,cat__deposit_type_No Deposit,0.094505
8,num__adr,0.086737
7,num__total_of_special_requests,0.080892
14,num__arrival_month,0.047070
4,num__previous_cancellations,0.042589
9,num__required_car_parking_spaces,0.038427
12,num__total_nights,0.034104
6,num__booking_changes,0.032813
